# NRPy in Ten Minutes

Follow one small pipeline from a symbolic expression to a registered C function and a generated-project command.

Navigation: [Index](../index.ipynb) | Previous: [Generated Project Anatomy](../0-getting_started/generated_projects.ipynb) | Next: [Parameters](params.ipynb)

## Why This Matters
The full workflow has many pieces, but the core idea is small: write algebra once, emit C, register a function, and let a generator assemble a project.

## What You Will See
You will inspect printed expressions, generated artifacts, or diagnostic tables and then connect them to the next notebook.

Prerequisite bridge: this notebook follows [Generated Project Anatomy](../0-getting_started/generated_projects.ipynb). Terms are defined here before they are used.

## One Miniature Pipeline
The expression below is a symbolic wave-energy density. NRPy turns it into C, then stores a full function in the registry.

In [1]:
import sympy as sp

import nrpy.c_codegen as ccg
from nrpy.c_function import CFunction_dict, register_CFunction
import nrpy.grid as grid
import nrpy.params as par

CFunction_dict.pop("overview_energy_density", None)
for name in ["uu", "vv"]:
    grid.glb_gridfcs_dict.pop(name, None)
par.set_parval_from_str("Infrastructure", "BHaH")

uu, vv = grid.register_gridfunctions(["uu", "vv"], group="EVOL")
wave_speed = par.register_CodeParameter("REAL", "tutorial_overview", "overview_wave_speed", 1.0)
energy_density = sp.Rational(1, 2) * (vv**2 + wave_speed**2 * uu**2)
assignment = ccg.c_codegen(energy_density, "energy_density", include_braces=False, verbose=False)
print(assignment)

register_CFunction(
    name="overview_energy_density",
    desc="Compute a compact wave-energy diagnostic.",
    cfunc_type="void",
    params="const double uu, const double vv, const double wave_speed, double *energy",
    body="*energy = 0.5*(vv*vv + wave_speed*wave_speed*uu*uu);\nreturn;",
)
print("registered keys:", sorted(CFunction_dict))
print(CFunction_dict["overview_energy_density"].function_prototype)
print("generator command used later: python -m nrpy.examples.wave_equation_cartesian")

energy_density = (1.0/2.0)*((overview_wave_speed)*(overview_wave_speed))*((uu)*(uu)) + (1.0/2.0)*((vv)*(vv));

registered keys: ['overview_energy_density']
void overview_energy_density(const double uu, const double vv, const double wave_speed, double *energy);
generator command used later: python -m nrpy.examples.wave_equation_cartesian


## Interpreting the Output
The output compresses the main workflow: register fields, build a symbolic expression, emit C, then register a complete C function. The named registry entry is what generated projects can write into source files.

## Where This Leads
- [Parameters](params.ipynb)
- [Indexed Expressions](indexedexp.ipynb)
- [Gridfunctions and Grid Metadata](grid.ipynb)
- [C Code Generation](c_codegen.ipynb)
- [C Function Registration](c_function.ipynb)
- [Wave Equation and C Code Generation](../3-wave_equation/wave_equation_and_c_codegen.ipynb)